In [4]:
import pandas as pd
import xml.etree.ElementTree as ET
import numpy as np
import os
import shutil


# =====================================================================
# =======================   PATIENTS VS HC  =========================
# =====================================================================


# ==========================
# PATHS
# ==========================
datasets_root = r"E:\King's College London\Department of Psychosis Shared Database Initiative - Documents\Analyses\SergioA6.1\datasets"
output_root = r"E:\King's College London\Department of Psychosis Shared Database Initiative - Documents\Analyses\SergioA6.1\data_extraction"

outcome_var = "outcome"

# ==========================
# LOAD AND MERGE BASE DATA
# ==========================
selected_vars = [
    "participant_id", "dataset", "session", "anat", "fmri", "dwi",
    "sex", "age", "group", "scanner"
]

df1 = pd.read_csv("filtered_selection_mdd.csv")[["participant_id", "session"]]
df11 = pd.read_csv("filtered_selection_hc.csv")[["participant_id", "session"]]
df1 = pd.concat([df1, df11], ignore_index=True)
df2 = pd.read_csv("metafile.csv")[selected_vars]
df = pd.merge(df1, df2, on=["participant_id", "session"], how="left")


# ==========================
# OUTCOME VAR
# ==========================
df["outcome"] = (df["group"]>0).astype(int)


# ==========================
# COVARIATES
# ==========================
df_covars = df[["participant_id", "session", "sex", "age", "scanner"]].copy()
scanner_dummies = pd.get_dummies(df_covars["scanner"], prefix="scanner")
df_covars = pd.concat([df_covars, scanner_dummies], axis=1)
scanner_mapping = {s: i + 1 for i, s in enumerate(df_covars["scanner"].unique())}
df_covars["scanner"] = df_covars["scanner"].map(scanner_mapping)

# ======================================
# EXPORT PATIENT SCANNER MAPPING 
# ======================================
scanner_mapping_patients = scanner_mapping.copy()
pd.Series(scanner_mapping_patients).to_csv("scanner_mapping.csv")
print("📤 Saved patient scanner mapping.")




# ==========================
# EXTRACT TIV & IQR FROM XML
# ==========================
def extract_tiv_iqr(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        vol_tiv_elems = root.findall(".//vol_TIV")
        vol_tiv = float(vol_tiv_elems[1].text) if len(vol_tiv_elems) >= 2 else np.nan
        iqr = np.nan
        for item in root.findall(".//item"):
            if item.text and "Image Quality Rating" in item.text:
                try:
                    iqr = float(item.text.split(":")[1].split("%")[0].strip())
                except Exception:
                    pass
                break
        return vol_tiv, iqr
    except Exception:
        return np.nan, np.nan

records = []
for _, row in df.iterrows():
    dataset, participant, session = row["dataset"], row["participant_id"], row["session"]
    xml_dir = os.path.join(datasets_root, str(dataset), "data", f"sub-{participant}", f"ses-{session}", "anat", "report")

    if not os.path.isdir(xml_dir):
        records.append({"participant_id": participant, "session": session, "dataset": dataset, "xml_file": None,
                        "xml_path": xml_dir, "vol_TIV": np.nan, "ImageQualityRating": np.nan})
        continue

    xml_files = [f for f in os.listdir(xml_dir) if f.lower().endswith(".xml")]
    if not xml_files:
        records.append({"participant_id": participant, "session": session, "dataset": dataset, "xml_file": None,
                        "xml_path": xml_dir, "vol_TIV": np.nan, "ImageQualityRating": np.nan})
        continue

    for xml_file in xml_files:
        xml_path = os.path.join(xml_dir, xml_file)
        vol_tiv, iqr = extract_tiv_iqr(xml_path)
        records.append({"participant_id": participant, "session": session, "dataset": dataset,
                        "xml_file": xml_file, "xml_path": xml_path, "vol_TIV": vol_tiv, "ImageQualityRating": iqr})

df_xml = pd.DataFrame(records)
df_xml = df_xml.dropna(subset=["ImageQualityRating"])
df_xml = df_xml.loc[df_xml.groupby(["participant_id", "session"])["ImageQualityRating"].idxmax()].reset_index(drop=True)
df_xml.rename(columns={"vol_TIV": "TIV", "ImageQualityRating": "IQR"}, inplace=True)

df_covars = df_covars.merge(df_xml[["participant_id", "session", "TIV", "IQR"]], on=["participant_id", "session"], how="left")

# ==========================
# BUILD MRI FILE PATHS FOR mwp1 & mwp2
# ==========================
mwp_datasets = []
for mwp in ["mwp1", "mwp2"]:
    paths = []
    for _, row in df_xml.iterrows():
        base = row["xml_file"].replace("cat_", "").replace(".xml", "")
        mri_file = f"{mwp}{base}.nii"
        mri_path = row["xml_path"].replace("report", "mri").replace(row["xml_file"], mri_file)
        paths.append({"participant_id": row["participant_id"], "session": row["session"], "dataset": row["dataset"],
                      "xml_file": row["xml_file"], "xml_path": row["xml_path"],
                      "mri_file": mri_file, "mri_path": mri_path, "mwp_type": mwp})
    mwp_datasets.append(pd.DataFrame(paths))

df_file_paths = pd.concat(mwp_datasets, ignore_index=True)

# ==========================
# COPY FILES (mwp1 & mwp2)
# ==========================
for mwp_type in ["mwp1", "mwp2"]:
    df_subset = df_file_paths[df_file_paths["mwp_type"] == mwp_type].copy()
    df_files_labels = df_subset.merge(
        df[["participant_id", "session", outcome_var]],
        on=["participant_id", "session"],
        how="left"
    )
    df_files_labels["file_exists"] = df_files_labels["mri_path"].apply(lambda x: os.path.isfile(x))
    df_files_labels = df_files_labels[df_files_labels["file_exists"]].copy()
    df_files_labels = df_files_labels[df_files_labels[outcome_var].isin([1, 0])].copy()

    base_folder = os.path.join(output_root, mwp_type)
    for subfolder in ["MDD", "HC"]:
        folder = os.path.join(base_folder, subfolder)
        os.makedirs(folder, exist_ok=True)
        for f in os.listdir(folder):
            os.remove(os.path.join(folder, f))

    copied_files = []
    for _, row in df_files_labels.iterrows():
        src = row["mri_path"]
        label = row[outcome_var]
        dest_folder = os.path.join(base_folder, "MDD" if label == 1 else "HC")
        dest_path = os.path.join(dest_folder, os.path.basename(src))
        shutil.copy2(src, dest_path)
        copied_files.append({
            "participant_id": row["participant_id"],
            "session": row["session"],
            "dataset": row["dataset"],
            "mri_path": src,
            "label": label,
            "mwp_type": mwp_type
        })

    df_used = pd.DataFrame(copied_files)
    print(f"✅ Copied {len(df_used)} labeled MRI files for {mwp_type}.")

    # ==========================
    # FINAL COVARIATES FILE (1:1 MATCH)
    # ==========================
    df_covars_final = df_used.merge(df_covars, on=["participant_id", "session"], how="left")
    df_covars_final["mri_file_name"] = df_covars_final["mri_path"].apply(lambda x: os.path.basename(x))

    dep_folder = os.path.join(base_folder, "MDD")
    no_dep_folder = os.path.join(base_folder, "HC")

    dep_files = sorted(os.listdir(dep_folder))
    no_dep_files = sorted(os.listdir(no_dep_folder))

    dep = df_covars_final[df_covars_final["mri_file_name"].isin(dep_files)]
    dep = dep.set_index("mri_file_name").loc[dep_files].reset_index()

    no_dep = df_covars_final[df_covars_final["mri_file_name"].isin(no_dep_files)]
    no_dep = no_dep.set_index("mri_file_name").loc[no_dep_files].reset_index()

    df_covars_ordered = pd.concat([dep, no_dep], ignore_index=True)

    covar_cols = ["participant_id", "session", "dataset", "sex", "age", "scanner", "TIV", "IQR", "label"] + \
                 [c for c in df_covars_ordered.columns if c.startswith("scanner_")]
    df_covars_ordered = df_covars_ordered[covar_cols]

    df_covars_ordered.to_excel(os.path.join(base_folder, "covariates.xlsx"), index=False)
    print(f"✅ Saved covariates ({len(df_covars_ordered)}) for {mwp_type} — order matches MDD/HC folders exactly.")




C:\Users\Sergio\AppData\Local\Temp\ipykernel_16704\1980786016.py:32: DtypeWarning: Columns (44,54,181) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("metafile.csv")[selected_vars]


📤 Saved patient scanner mapping.
✅ Copied 2034 labeled MRI files for mwp1.
✅ Saved covariates (2034) for mwp1 — order matches MDD/HC folders exactly.
✅ Copied 2034 labeled MRI files for mwp2.
✅ Saved covariates (2034) for mwp2 — order matches MDD/HC folders exactly.
